# Testing Local Models

## Paddle OCR Standard

In [57]:
from paddleocr import PaddleOCR
import numpy as np
from PIL import Image
import cv2
import pymupdf

In [58]:
paddle_ocr = PaddleOCR(
    lang="en",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    text_detection_model_name="PP-OCRv6_small_det",
    text_recognition_model_name="PP-OCRv6_small_rec",
    cpu_threads=5,        
    enable_mkldnn=False,
)

C:\Users\User\AppData\Local\Temp\ipykernel_12440\3334214711.py:1: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  paddle_ocr = PaddleOCR(
c:\Users\User\.conda\envs\autograde_ai\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\User\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_small_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\User\.paddlex\official_models\PP-OCRv6_small_det`.
Creating model: ('PP

In [59]:
image = Image.open("test_documents/mild_handwriting_paper.png").convert("RGB")
image_array = np.asarray(image)
bgr_img_array = cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR)

In [60]:
result = paddle_ocr.predict(image_array, use_textline_orientation=True)

In [61]:
text = " ".join([" ".join(page["rec_texts"]) for page in result])

In [62]:
text

'21. Superuised learning is use fo predict He out come by prouiding data to be frain and test. Example: linear regresion navic byes,khearst. 25).T mr appation a n are ae,healt avr. 26)- Struetured data  type of data teat are aell label Like cu Json  Semi- Stractured data is atype of data that is partialy label. 27). To find out an Outlier in data ae can use Uisuclizatron techige like Scatter plot or Box plot and with anotler methods. 28). Data pre processing is a type of data that are wag to processing and dleaning daba. 24). Elucidate is a way to calculate He dotanee in kNN. Exampe: Data normalizabion is reqauived becaere inbalance data can 29. effeat the result f. 31). k in k-means algorith stand for k 32). Some hypexparameters of machire learning algorithms: -epoch'

#### From PDF

In [73]:
pdf = pymupdf.open("test_documents/handwritten_exam_1.pdf")

In [74]:
text = ""
for page in pdf.pages():
    pix = page.get_pixmap(dpi=200)
    image_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
    result = paddle_ocr.predict(image_array, use_textline_orientation=True)
    page_text = " ".join([" ".join(page["rec_texts"]) for page in result])
    text += page_text

Resized image size (4960x7014) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4960x7014) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4960x7014) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (4960x7014) exceeds max_side_limit of 4000. Resizing to fit within limit.


In [75]:
text

"Solution 1. C 2. A 3.D 4.① S. n G.B 7. B 8. A 5. D 10. B 11. C 12. C 13.B 14.C 15. C 16. C B 18. C 19.B 90. C.Supervised Learning 21. dataset from input (teature) into output (taget) Variable. 23. model Pelationship between dependent Variable ie use regression analysis to predict the numberical of Value best on theire Variable. 25. Two major applications of data science. Hospitalities Agricalture Distinguish between the terms : 26. Semi-Structured data Structured data - full of Orgenization - Orgonization format Such as HTML, XML, JSON format -Commputer Can read as use Computer Can read cnd ITML, XML, JSON analyst. Need to do pre-proccessing - No need to do data - Before analyst need to do proccessing. Can take data and pre-proccess of data. analyst - Format in Rational Format in Rational data base database.31. k is numberical of cluster... Two challenges of Machine learnong: 24. -Supervise learning, it's predict on Lable of varrable target.Now cwe point on three different sum classi 

## TrOCR

In [81]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

In [82]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

Loading weights: 100%|██████████| 478/478 [00:00<00:00, 1027.75it/s]
[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [83]:
image = Image.open("test_documents/mild_handwriting_paper.png").convert("RGB")
image = image.crop((0, 10, image.size[0], 40))
pixel_values = processor(image, return_tensors="pt").to(model.device).pixel_values
generated_ids = model.generate(pixel_values)

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


c:\Users\User\.conda\envs\autograde_ai\Lib\site-packages\transformers\generation\utils.py:1954: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [84]:
generated_text

'a1 . I supervised learning to protect the oak come by providing , in'

## PaddleOCR + PARSeq Pipeline

### For image

In [ ]:
import cv2
import numpy as np
import torch
from PIL import Image
from paddleocr import PaddleOCR

In [ ]:
class PaddlePARSeqPipeline:

    def __init__(self, paddle_detector: PaddleOCR, parseq_model: torch.nn.Module):
        self.detector = paddle_detector
        self.recognizer = parseq_model

    def _crop_and_align(self, image_np: np.ndarray, bbox):
        pts = np.array(bbox, dtype=np.float32)

        width = int(
            max(
                np.linalg.norm(pts[0] - pts[1]), np.linalg.norm(pts[2] - pts[3])
            )
        )
        height = int(
            max(
                np.linalg.norm(pts[0] - pts[3]), np.linalg.norm(pts[1] - pts[2])
            )
        )

        dst_pts = np.array(
            [[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32
        )
        matrix = cv2.getPerspectiveTransform(pts, dst_pts)
        cropped = cv2.warpPerspective(image_np, matrix, (width, height))

        return Image.fromarray(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))

    def predict(self, image_np: np.ndarray, batch_size: int=8):
    
        det_results = self.detector.predict(image_np)
        boxes = det_results[0] if det_results and det_results[0] else []

        if not boxes:
            return []

        crops = []
        for box in boxes:
            crop = self._crop_and_align(image_np, box)
            if crop is not None:
                crops.append(crop)

        results = []
        for i in range(0, len(crops), batch_size):
            batch_crops = crops[i : i + batch_size]
            tensors = torch.stack(
                [self.img_transform(c) for c in batch_crops]
            )

            with torch.inference_mode():
                logits = self.parseq(tensors)
                preds = logits.softmax(-1)
                labels, confidences = self.parseq.tokenizer.decode(preds)

            for j, (label, conf) in enumerate(zip(labels, confidences)):
                avg_conf = conf.mean().item()
                results.append(
                    {
                        "text": label,
                        "confidence": round(avg_conf, 4),
                    }
                )

        return results

In [ ]:
detector = PaddleOCR(
    use_angle_cls=True, lang="en"
)

parseq = (
    torch.hub.load("baudm/parseq", "parseq", pretrained=True)
    .eval()
)

pipeline = PaddlePARSeqPipeline(detector, parseq)

In [ ]:
image = cv2.imread("test_documents/good_handwriting_paper.png")
output = pipeline.predict(image, batch_size=8)

for item in output:
    print(f"{item['text']} (Conf: {item['confidence']})")

## Results

# Hosted Models

## LlamaIndex

In [68]:
from llama_cloud import LlamaCloud

In [69]:
client = LlamaCloud()

In [70]:
file = client.files.create(file="test_documents/bad_handwriting_paper.png", purpose="parse")
result = client.parsing.parse(
    file_id=file.id,
    tier="cost_effective",
    version="latest",
    expand=["markdown"],
)

In [71]:
text = ""
for page in result.markdown.pages:
    text += page.markdown
text

'Hello r/Handwriting!\n\nWe stood there in that world for what ~~felt~~ like a long time. The second ~~the~~ I started to say something, Lindsay did too, so we laughed weirdly and stood there some more.\n\n"I hope you really do write to me."\n\nI finally said as I hugged Lindsay all fast and awkward like I used to hug teachers on the last day of class when I was little and there would be a whole line of students ~~students~~ behind me also waiting to get a hug, and we were all shy and embarrassed about the whole thing.'

## Results

Test various local & hodsted models on **Local system & Google Collab**  
### Requirements
- Fast enough to run on local system (CPU only)
- Good accuracy for good to mild handwriting

### Local Models tested
These are results for local models tested:
1. **PaddleOCR** Fast but poor accuracy on mildly bad handwriting
2. **TrOCR & GLMOCR** Good accuracy but requried cropping of each line which significantly increased the time
3. **PaddleOCR** Didn't test because it is notorious for being slow
4. **PaddleOCR + PARSeq Pipeline** Good accuracy but slower speed than required

### Hosted Models
Tested only **LLamaIndex** which gave fast and accurate performance + OMR.

### Conclusion 
The local models for current setup (CPU only) can't work reliably.
LlamaIndex is chosen for document reading.

# Document Reader

In [77]:
class DocumentReader:

    def __init__(self, llama_cloud: LlamaCloud):
        self.client = llama_cloud

    def read_doc(self, file_bytes: bytes):
        file = client.files.create(file=file_bytes, purpose="parse")
        result = client.parsing.parse(
            file_id=file.id,
            tier="cost_effective",
            version="latest",
            expand=["markdown"],
        )
        text = ""
        for page in result.markdown.pages:
            text += page.markdown
        return text

In [78]:
document_reader = DocumentReader(LlamaCloud())

In [79]:
pdf = pymupdf.open("test_documents/handwritten_exam_1.pdf")

In [80]:
document_reader.read_doc(pdf.tobytes())

"<u>Solution</u>\n\n<table>\n  <tr>\n    <th>#</th>\n    <th>Answer</th>\n  </tr>\n<tr>\n    <td>1.</td>\n    <td>C</td>\n  </tr>\n<tr>\n    <td>2.</td>\n    <td>A</td>\n  </tr>\n<tr>\n    <td>3.</td>\n    <td>D</td>\n  </tr>\n<tr>\n    <td>4.</td>\n    <td>D</td>\n  </tr>\n<tr>\n    <td>5.</td>\n    <td>A</td>\n  </tr>\n<tr>\n    <td>6.</td>\n    <td>B</td>\n  </tr>\n<tr>\n    <td>7.</td>\n    <td>B</td>\n  </tr>\n<tr>\n    <td>8.</td>\n    <td>A</td>\n  </tr>\n<tr>\n    <td>9.</td>\n    <td>D</td>\n  </tr>\n<tr>\n    <td>10.</td>\n    <td>B</td>\n  </tr>\n<tr>\n    <td>11.</td>\n    <td>C</td>\n  </tr>\n<tr>\n    <td>12.</td>\n    <td>C</td>\n  </tr>\n<tr>\n    <td>13.</td>\n    <td>B</td>\n  </tr>\n<tr>\n    <td>14.</td>\n    <td>C</td>\n  </tr>\n<tr>\n    <td>15.</td>\n    <td>C</td>\n  </tr>\n<tr>\n    <td>16.</td>\n    <td>C</td>\n  </tr>\n<tr>\n    <td>17.</td>\n    <td>B</td>\n  </tr>\n<tr>\n    <td>18.</td>\n    <td>C</td>\n  </tr>\n<tr>\n    <td>19.</td>\n    <td>B</td>\n  </